# LectureAI Video Analiz Pipeline (GPU)
Bucket'taki ders videosunu GPU ile analiz eder, sonuc JSON'ini bucket'a kaydeder.

In [ ]:
import os, sys, json
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

if IN_COLAB:
    # --- Colab: credential dosyasini yukle ---
    CREDENTIAL_PATH = '/content/service_account.json'
    if not os.path.exists(CREDENTIAL_PATH):
        from google.colab import files
        print('Service account JSON dosyasini yukleyin:')
        uploaded = files.upload()
        for fname, content in uploaded.items():
            CREDENTIAL_PATH = f'/content/{fname}'
            with open(CREDENTIAL_PATH, 'wb') as f:
                f.write(content)
            break
    REPO_DIR = '/content/LectureAI'
    WORK_DIR = '/content/work'
else:
    SCRIPT_DIR = Path(os.path.abspath('')).resolve()
    REPO_DIR = str(SCRIPT_DIR.parent) if SCRIPT_DIR.name == 'video' else str(SCRIPT_DIR)
    creds = list(Path(REPO_DIR).glob('senior-design-*.json'))
    if not creds:
        creds = list((Path(REPO_DIR) / 'web-app' / 'backend').glob('service_account.json'))
    CREDENTIAL_PATH = str(creds[0]) if creds else ''
    WORK_DIR = str(Path(REPO_DIR) / 'work')

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = CREDENTIAL_PATH
print(f'Ortam: {"Colab" if IN_COLAB else "Lokal"}')
print(f'Credential: {CREDENTIAL_PATH}')

In [ ]:
import subprocess
if IN_COLAB:
    REPO_URL = 'https://github.com/pinaraltinok/LectureAI.git'
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'numpy<2', 'mediapipe==0.10.21', 'easyocr==1.7.2',
        'google-cloud-storage>=2.16.0', 'opencv-python-headless<4.10.0',
        'pandas<2.2.0', 'torch', 'matplotlib'], check=True)
    import site, importlib
    importlib.reload(site)

sys.path.insert(0, REPO_DIR)

# GPU kontrol
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('UYARI: GPU bulunamadi, CPU ile calisacak (yavas)')
print('Ortam hazir.')

In [ ]:
import time
import pandas as pd
from google.cloud import storage
from video.dynamic_visual_pipeline import run_dynamic_visual_poc
from video.frame_extractor import get_video_meta

client = storage.Client()
print('Pipeline ve GCS client hazir.')

## Konfigurasyon
Asagidaki degerleri degistirin. Bucket'taki mevcut videolar listelenir.

In [ ]:
# === BUCKET'TAKI VIDEOLARI LISTELE ===
bucket = client.bucket('lectureai_full_videos')
blobs = list(bucket.list_blobs(prefix='Lesson_Records/'))
videos = [b.name for b in blobs if b.name.lower().endswith('.mp4')]
print('Bucket\"taki videolar:')
for i, v in enumerate(videos):
    size_mb = [b for b in blobs if b.name == v][0].size / 1024 / 1024
    print(f'  [{i}] gs://lectureai_full_videos/{v}  ({size_mb:.0f} MB)')

In [ ]:
# ==========================================
#  KONFIGURASYON
#  Colab: Asagidaki degerleri doldurun
#  Backend: Env var'lardan otomatik okunur
# ==========================================

# Yukaridaki listeden video secin veya tam URI yazin
GCS_VIDEO_URI = os.environ.get('LECTUREAI_VIDEO_URI', '') or \
    'gs://lectureai_full_videos/Lesson_Records/TURPRM40W273_TUE-1930_8-9(L0).mp4'

# Ogretmen adi
TEACHER_NAME = os.environ.get('LECTUREAI_TEACHER_NAME', '') or \
    'Zehra Bozkurt'

# Sonuc bucket
RESULT_BUCKET = 'lectureai_processed'

# Analiz parametreleri
ANALYSIS_INTERVAL_SEC = 4.0
RELOCALIZE_INTERVAL_SEC = 60.0
SMILE_THRESHOLD = 0.60

# ==========================================

def parse_gcs_uri(uri):
    if uri.startswith('gs://'):
        parts = uri[5:].split('/', 1)
        return parts[0], parts[1]
    elif uri.startswith('https://storage.googleapis.com/'):
        parts = uri.replace('https://storage.googleapis.com/', '').split('/', 1)
        return parts[0], parts[1]
    raise ValueError(f'Gecersiz GCS URI: {uri}')

VIDEO_BUCKET, VIDEO_BLOB_PATH = parse_gcs_uri(GCS_VIDEO_URI)
VIDEO_FILENAME = Path(VIDEO_BLOB_PATH).name
VIDEO_STEM = Path(VIDEO_FILENAME).stem
RESULT_PREFIX = 'results/' + str(Path(VIDEO_BLOB_PATH).parent) + '/'

LOCAL_VIDEO_DIR = os.path.join(WORK_DIR, 'videos')
LOCAL_OUTPUT_DIR = os.path.join(WORK_DIR, 'outputs')
os.makedirs(LOCAL_VIDEO_DIR, exist_ok=True)
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)

print(f'Video: {GCS_VIDEO_URI}')
print(f'Ogretmen: {TEACHER_NAME}')
print(f'Sonuc: gs://{RESULT_BUCKET}/{RESULT_PREFIX}')

In [ ]:
# === VIDEO INDIRME ===
local_video_path = os.path.join(LOCAL_VIDEO_DIR, VIDEO_FILENAME)

if not os.path.exists(local_video_path):
    print(f'Video indiriliyor: {GCS_VIDEO_URI}')
    t0 = time.time()
    bucket = client.bucket(VIDEO_BUCKET)
    blob = bucket.blob(VIDEO_BLOB_PATH)
    blob.download_to_filename(local_video_path)
    size_mb = os.path.getsize(local_video_path) / 1024 / 1024
    print(f'Indirildi: {time.time()-t0:.1f} sn, {size_mb:.1f} MB')
else:
    print(f'Video zaten mevcut: {local_video_path}')

meta = get_video_meta(local_video_path)
print(f'\nVideo meta:')
for k, v in meta.items():
    print(f'  {k}: {v}')

In [ ]:
# === ANALIZ (GPU) ===
print(f'Analiz basliyor: {VIDEO_FILENAME}')
print(f'  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'  interval={ANALYSIS_INTERVAL_SEC}s, relocalize={RELOCALIZE_INTERVAL_SEC}s')
print('=' * 60)

t0 = time.time()
summary, debug_df = run_dynamic_visual_poc(
    video_path=local_video_path,
    teacher_name=TEACHER_NAME,
    analysis_interval_sec=ANALYSIS_INTERVAL_SEC,
    relocalize_interval_sec=RELOCALIZE_INTERVAL_SEC,
    smile_threshold=SMILE_THRESHOLD,
    start_sec=0.0,
    end_sec=None,
    only_camera_open_frames=True,
    debug_dir=None,
)
elapsed = time.time() - t0
print(f'\nAnaliz tamamlandi: {elapsed:.1f} sn ({elapsed/60:.1f} dk)')

summary.update({
    'source_uri': GCS_VIDEO_URI,
    'video_filename': VIDEO_FILENAME,
    'teacher_name': TEACHER_NAME,
    'video_duration_sec': meta['duration_sec'],
    'video_fps': meta['fps'],
    'video_width': meta['width'],
    'video_height': meta['height'],
})

print('\nOzet:')
for k, v in summary.items():
    print(f'  {k}: {v}')

In [ ]:
# === RAPOR OLUSTURMA & GCS'YE YUKLEME ===
full_report = {
    'source_uri': GCS_VIDEO_URI,
    'teacher_name': TEACHER_NAME,
    'config': {
        'analysis_interval_sec': ANALYSIS_INTERVAL_SEC,
        'relocalize_interval_sec': RELOCALIZE_INTERVAL_SEC,
        'smile_threshold': SMILE_THRESHOLD,
    },
    'video_meta': meta,
    'summary': summary,
}

output_dir = os.path.join(LOCAL_OUTPUT_DIR, VIDEO_STEM)
os.makedirs(output_dir, exist_ok=True)

report_path = os.path.join(output_dir, 'lecture_report.json')
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(full_report, f, ensure_ascii=False, indent=2)

debug_csv_path = os.path.join(output_dir, 'debug.csv')
debug_df.to_csv(debug_csv_path, index=False)

print(f'Lokal rapor: {report_path}')
print(f'Boyut: {os.path.getsize(report_path)/1024:.1f} KB')

# GCS'ye yukle
try:
    result_bucket = client.bucket(RESULT_BUCKET)

    gcs_report = f'{RESULT_PREFIX}lecture_report.json'
    blob = result_bucket.blob(gcs_report)
    blob.upload_from_filename(report_path)
    print(f'\nGCS yuklendi: gs://{RESULT_BUCKET}/{gcs_report}')

    gcs_debug = f'{RESULT_PREFIX}debug.csv'
    blob = result_bucket.blob(gcs_debug)
    blob.upload_from_filename(debug_csv_path)
    print(f'GCS yuklendi: gs://{RESULT_BUCKET}/{gcs_debug}')
except Exception as e:
    print(f'GCS yukleme hatasi: {e}')

if IN_COLAB:
    try:
        from google.colab import files
        files.download(report_path)
    except Exception:
        pass

In [ ]:
# === SONUCLARI GORUNTULE ===
print('=' * 60)
print('ANALIZ OZETI')
print('=' * 60)
for k, v in summary.items():
    print(f'  {k}: {v}')

print('\n' + '=' * 60)
print('DEBUG DF (ilk 20 satir)')
print('=' * 60)
display(debug_df.head(20))

In [ ]:
# === GORSEL DASHBOARD ===
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0f0f23')
fig.suptitle(f'LectureAI - {TEACHER_NAME} - {VIDEO_FILENAME}',
             color='white', fontsize=14, fontweight='bold')

kpi_labels = ['Ogretmen\nBulunma', 'Kamera Acik', 'Gulumseme', 'El Gorunur', 'Hareket']
kpi_values = [
    summary.get('teacher_locate_ratio', 0) * 100,
    summary.get('camera_open_ratio_total', 0) * 100,
    summary.get('smile_frame_ratio', 0) * 100,
    summary.get('hand_visible_ratio', 0) * 100,
    summary.get('movement_energy_avg', 0) * 100,
]

ax1 = axes[0]
ax1.set_facecolor('#1a1a2e')
colors = ['#e94560', '#0f3460', '#533483', '#16213e', '#e94560']
bars = ax1.bar(kpi_labels, kpi_values, color=colors, edgecolor='white', linewidth=0.5)
for bar, v in zip(bars, kpi_values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{v:.1f}%', ha='center', color='white', fontsize=9)
ax1.set_title('Metrik Oranlari', color='white', fontsize=12, fontweight='bold')
ax1.tick_params(colors='white')
ax1.set_ylim(0, 105)

ax2 = axes[1]
ax2.set_facecolor('#1a1a2e')
if not debug_df.empty and 't_sec' in debug_df.columns:
    used = debug_df[debug_df.get('used_for_metrics', pd.Series([False])) == True]
    if not used.empty and 'smile_score' in used.columns:
        ax2.plot(used['t_sec'], used['smile_score'], 'o-', color='#e94560',
                 linewidth=1, markersize=3, alpha=0.8)
        ax2.set_ylabel('Smile Score', color='white')
ax2.set_xlabel('Zaman (sn)', color='white')
ax2.set_title('Gulumseme Trendi', color='white', fontsize=12, fontweight='bold')
ax2.tick_params(colors='white')

ax3 = axes[2]
ax3.set_facecolor('#1a1a2e')
if not debug_df.empty and 't_sec' in debug_df.columns:
    used = debug_df[debug_df.get('used_for_metrics', pd.Series([False])) == True]
    if not used.empty and 'movement_energy' in used.columns:
        ax3.fill_between(used['t_sec'], used['movement_energy'], alpha=0.4, color='#533483')
        ax3.plot(used['t_sec'], used['movement_energy'], color='#533483', linewidth=1)
ax3.set_xlabel('Zaman (sn)', color='white')
ax3.set_title('Hareket Enerjisi', color='white', fontsize=12, fontweight='bold')
ax3.tick_params(colors='white')

plt.tight_layout()
dashboard_path = os.path.join(output_dir, 'dashboard.png')
plt.savefig(dashboard_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Dashboard: {dashboard_path}')